# Order Book Visualization and Animation
This notebook visualizes order book snapshots and creates an animation of the order book over time using Plotly.

In [ ]:
import json
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import numpy as np

import sys
import os

root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(root_path)



In [ ]:
# Configuration
exchange = "binance"
trading_pair = "USDT-BRL"
current_date = datetime.now().strftime("%Y-%m-%d")
file_path = f"{exchange}_{trading_pair}_order_book_snapshots_{current_date}.txt"
full_path = os.path.join(root_path, 'data', 'market_data', file_path)
print(f"Loading data from: {file_path}")

In [ ]:
# Load order book snapshots
snapshots = []
with open(full_path, 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            snapshots.append(json.loads(line))

print(f"Loaded {len(snapshots)} snapshots")
print(f"Time range: {snapshots[0]['ts']:.2f} to {snapshots[-1]['ts']:.2f}")
print(f"Duration: {snapshots[-1]['ts'] - snapshots[0]['ts']:.2f} seconds")

In [ ]:
# Function to create order book figure
def create_orderbook_figure(snapshot, title=None):
    # Extract bids and asks
    bids = np.array(snapshot['bids'])
    asks = np.array(snapshot['asks'])
    
    # Calculate cumulative volumes
    bid_prices = bids[:, 0]
    bid_volumes = bids[:, 1]
    bid_cumulative = np.cumsum(bid_volumes[::-1])[::-1]
    
    ask_prices = asks[:, 0]
    ask_volumes = asks[:, 1]
    ask_cumulative = np.cumsum(ask_volumes)
    
    # Calculate mid price
    best_bid = bid_prices[0]
    best_ask = ask_prices[0]
    mid_price = (best_bid + best_ask) / 2
    spread = best_ask - best_bid
    
    # Create figure
    fig = go.Figure()
    
    # Add bids (buy orders) in green
    fig.add_trace(go.Scatter(
        x=bid_prices,
        y=bid_cumulative,
        fill='tozeroy',
        fillcolor='rgba(0, 255, 0, 0.3)',
        line=dict(color='darkgreen', width=2, shape='hv'),
        name='Bids',
        mode='lines',
        hovertemplate='Price: %{x:.4f}<br>Cumulative Volume: %{y:.2f}<extra></extra>'
    ))
    
    # Add asks (sell orders) in red
    fig.add_trace(go.Scatter(
        x=ask_prices,
        y=ask_cumulative,
        fill='tozeroy',
        fillcolor='rgba(255, 0, 0, 0.3)',
        line=dict(color='darkred', width=2, shape='hv'),
        name='Asks',
        mode='lines',
        hovertemplate='Price: %{x:.4f}<br>Cumulative Volume: %{y:.2f}<extra></extra>'
    ))
    
    # Add mid price line
    fig.add_vline(
        x=mid_price,
        line_dash="dash",
        line_color="black",
        opacity=0.5,
        annotation_text=f"Mid: {mid_price:.4f}",
        annotation_position="top"
    )
    
    # Update layout
    if title:
        fig_title = title
    else:
        timestamp = datetime.fromtimestamp(snapshot['ts']).strftime('%Y-%m-%d %H:%M:%S')
        fig_title = f'{trading_pair} Order Book @ {timestamp}<br><sub>Mid: {mid_price:.4f} | Spread: {spread:.4f}</sub>'
    
    fig.update_layout(
        title=dict(text=fig_title, x=0.5, xanchor='center'),
        xaxis_title='Price',
        yaxis_title='Cumulative Volume',
        hovermode='x unified',
        template='plotly_white',
        height=600,
        showlegend=True,
        legend=dict(x=0.01, y=0.99, xanchor='left', yanchor='top')
    )
    
    return fig

In [ ]:
# Plot the first snapshot
fig = create_orderbook_figure(snapshots[0])
fig.show()

In [ ]:
# Plot the latest snapshot
fig = create_orderbook_figure(snapshots[-1])
fig.show()

## Order Book Animation
Create an animated visualization of the order book over time using Plotly's animation features.

In [ ]:
# Animation parameters
frame_skip = max(1, len(snapshots) // 200)  # Limit to ~200 frames for performance
snapshots_to_animate = snapshots[::frame_skip]

print(f"Creating animation with {len(snapshots_to_animate)} frames (skipping every {frame_skip} snapshots)")

In [ ]:
# Create animated order book
frames = []
all_prices = []
all_volumes = []

# First pass: collect data for consistent axes
for snapshot in snapshots_to_animate:
    bids = np.array(snapshot['bids'])
    asks = np.array(snapshot['asks'])
    all_prices.extend(bids[:, 0].tolist())
    all_prices.extend(asks[:, 0].tolist())
    all_volumes.append(sum(bids[:, 1]) + sum(asks[:, 1]))

price_min, price_max = min(all_prices), max(all_prices)
volume_max = max(all_volumes)
price_padding = (price_max - price_min) * 0.05

# Second pass: create frames
for i, snapshot in enumerate(snapshots_to_animate):
    bids = np.array(snapshot['bids'])
    asks = np.array(snapshot['asks'])
    
    bid_prices = bids[:, 0]
    bid_volumes = bids[:, 1]
    bid_cumulative = np.cumsum(bid_volumes[::-1])[::-1]
    
    ask_prices = asks[:, 0]
    ask_volumes = asks[:, 1]
    ask_cumulative = np.cumsum(ask_volumes)
    
    mid_price = (bid_prices[0] + ask_prices[0]) / 2
    spread = ask_prices[0] - bid_prices[0]
    timestamp = datetime.fromtimestamp(snapshot['ts']).strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]
    
    frame = go.Frame(
        data=[
            go.Scatter(
                x=bid_prices,
                y=bid_cumulative,
                fill='tozeroy',
                fillcolor='rgba(0, 255, 0, 0.3)',
                line=dict(color='darkgreen', width=2, shape='hv'),
                name='Bids',
                mode='lines'
            ),
            go.Scatter(
                x=ask_prices,
                y=ask_cumulative,
                fill='tozeroy',
                fillcolor='rgba(255, 0, 0, 0.3)',
                line=dict(color='darkred', width=2, shape='hv'),
                name='Asks',
                mode='lines'
            )
        ],
        name=str(i),
        layout=go.Layout(
            title_text=f'{trading_pair} Order Book | {timestamp}<br><sub>Mid: {mid_price:.4f} | Spread: {spread:.6f} | Frame {i+1}/{len(snapshots_to_animate)}</sub>'
        )
    )
    frames.append(frame)

# Create initial figure with first snapshot
first_snapshot = snapshots_to_animate[0]
bids = np.array(first_snapshot['bids'])
asks = np.array(first_snapshot['asks'])

bid_prices = bids[:, 0]
bid_volumes = bids[:, 1]
bid_cumulative = np.cumsum(bid_volumes[::-1])[::-1]

ask_prices = asks[:, 0]
ask_volumes = asks[:, 1]
ask_cumulative = np.cumsum(ask_volumes)

mid_price = (bid_prices[0] + ask_prices[0]) / 2
spread = ask_prices[0] - bid_prices[0]
timestamp = datetime.fromtimestamp(first_snapshot['ts']).strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]

fig = go.Figure(
    data=[
        go.Scatter(
            x=bid_prices,
            y=bid_cumulative,
            fill='tozeroy',
            fillcolor='rgba(0, 255, 0, 0.3)',
            line=dict(color='darkgreen', width=2, shape='hv'),
            name='Bids',
            mode='lines',
            hovertemplate='Price: %{x:.4f}<br>Cumulative Volume: %{y:.2f}<extra></extra>'
        ),
        go.Scatter(
            x=ask_prices,
            y=ask_cumulative,
            fill='tozeroy',
            fillcolor='rgba(255, 0, 0, 0.3)',
            line=dict(color='darkred', width=2, shape='hv'),
            name='Asks',
            mode='lines',
            hovertemplate='Price: %{x:.4f}<br>Cumulative Volume: %{y:.2f}<extra></extra>'
        )
    ],
    frames=frames
)

# Update layout with animation controls
fig.update_layout(
    title=dict(
        text=f'{trading_pair} Order Book | {timestamp}<br><sub>Mid: {mid_price:.4f} | Spread: {spread:.6f} | Frame 1/{len(snapshots_to_animate)}</sub>',
        x=0.5,
        xanchor='center'
    ),
    xaxis=dict(
        title='Price',
        range=[price_min - price_padding, price_max + price_padding]
    ),
    yaxis=dict(
        title='Cumulative Volume',
        range=[0, volume_max * 1.1]
    ),
    hovermode='x unified',
    template='plotly_white',
    height=700,
    showlegend=True,
    updatemenus=[
        dict(
            type='buttons',
            showactive=False,
            buttons=[
                dict(
                    label='Play',
                    method='animate',
                    args=[None, dict(
                        frame=dict(duration=50, redraw=True),
                        fromcurrent=True,
                        mode='immediate',
                        transition=dict(duration=0)
                    )]
                ),
                dict(
                    label='Pause',
                    method='animate',
                    args=[[None], dict(
                        frame=dict(duration=0, redraw=False),
                        mode='immediate',
                        transition=dict(duration=0)
                    )]
                )
            ],
            x=0.1,
            y=1.15,
            xanchor='left',
            yanchor='top'
        )
    ],
    sliders=[
        dict(
            active=0,
            steps=[
                dict(
                    args=[[f.name], dict(
                        frame=dict(duration=0, redraw=True),
                        mode='immediate',
                        transition=dict(duration=0)
                    )],
                    label=str(i+1),
                    method='animate'
                )
                for i, f in enumerate(frames)
            ],
            x=0.1,
            y=0,
            len=0.9,
            xanchor='left',
            yanchor='top',
            pad=dict(b=10, t=50)
        )
    ]
)

fig.show()

print("\nAnimation created! Use the Play button to start or drag the slider to navigate frames.")

## Save Animation to File
Save the animation as an interactive HTML file.

In [ ]:
# Save as interactive HTML
output_file = f"{exchange}_{trading_pair}_orderbook_animation_{current_date}.html"
fig.write_html(output_file)
print(f"Interactive animation saved to: {output_file}")

## Order Book Statistics
Visualize key metrics over time.

In [ ]:
# Calculate statistics across all snapshots
timestamps = []
mid_prices = []
spreads = []
bid_volumes = []
ask_volumes = []

for snapshot in snapshots:
    bids = np.array(snapshot['bids'])
    asks = np.array(snapshot['asks'])
    
    best_bid = bids[0, 0]
    best_ask = asks[0, 0]
    
    timestamps.append(datetime.fromtimestamp(snapshot['ts']))
    mid_prices.append((best_bid + best_ask) / 2)
    spreads.append(best_ask - best_bid)
    bid_volumes.append(sum(bids[:, 1]))
    ask_volumes.append(sum(asks[:, 1]))

# Create subplots
fig_stats = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Mid Price Over Time', 'Spread Over Time', 
                    'Total Volumes Over Time', 'Spread Distribution'),
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}],
           [{'type': 'scatter'}, {'type': 'histogram'}]]
)

# Mid price over time
fig_stats.add_trace(
    go.Scatter(x=timestamps, y=mid_prices, mode='lines', name='Mid Price', line=dict(width=1)),
    row=1, col=1
)

# Spread over time
fig_stats.add_trace(
    go.Scatter(x=timestamps, y=spreads, mode='lines', name='Spread', line=dict(color='orange', width=1)),
    row=1, col=2
)

# Total volumes
fig_stats.add_trace(
    go.Scatter(x=timestamps, y=bid_volumes, mode='lines', name='Bid Volume', 
               line=dict(color='green', width=1), opacity=0.7),
    row=2, col=1
)
fig_stats.add_trace(
    go.Scatter(x=timestamps, y=ask_volumes, mode='lines', name='Ask Volume', 
               line=dict(color='red', width=1), opacity=0.7),
    row=2, col=1
)

# Spread distribution
fig_stats.add_trace(
    go.Histogram(x=spreads, nbinsx=50, name='Spread Distribution', marker=dict(color='lightblue')),
    row=2, col=2
)

# Update axes
fig_stats.update_xaxes(title_text='Time', row=1, col=1)
fig_stats.update_xaxes(title_text='Time', row=1, col=2)
fig_stats.update_xaxes(title_text='Time', row=2, col=1)
fig_stats.update_xaxes(title_text='Spread', row=2, col=2)

fig_stats.update_yaxes(title_text='Price', row=1, col=1)
fig_stats.update_yaxes(title_text='Spread', row=1, col=2)
fig_stats.update_yaxes(title_text='Volume', row=2, col=1)
fig_stats.update_yaxes(title_text='Frequency', row=2, col=2)

# Update layout
fig_stats.update_layout(
    height=900,
    showlegend=True,
    template='plotly_white',
    title_text=f'{trading_pair} Order Book Statistics'
)

fig_stats.show()

# Print summary statistics
print("\n=== Order Book Statistics ===")
print(f"Mid Price - Mean: {np.mean(mid_prices):.4f}, Std: {np.std(mid_prices):.4f}")
print(f"Mid Price - Min: {np.min(mid_prices):.4f}, Max: {np.max(mid_prices):.4f}")
print(f"\nSpread - Mean: {np.mean(spreads):.6f}, Std: {np.std(spreads):.6f}")
print(f"Spread - Min: {np.min(spreads):.6f}, Max: {np.max(spreads):.6f}")
print(f"\nBid Volume - Mean: {np.mean(bid_volumes):.2f}, Std: {np.std(bid_volumes):.2f}")
print(f"Ask Volume - Mean: {np.mean(ask_volumes):.2f}, Std: {np.std(ask_volumes):.2f}")

In [ ]:
# Visualize storage metrics
fig_storage = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Line Size Distribution', 'Time Interval Distribution', 
                    'Storage Growth Projection', 'Cumulative File Size Over Time'),
    specs=[[{'type': 'histogram'}, {'type': 'histogram'}],
           [{'type': 'bar'}, {'type': 'scatter'}]]
)

# Line size distribution
fig_storage.add_trace(
    go.Histogram(x=line_sizes, nbinsx=50, name='Line Size', marker=dict(color='blue')),
    row=1, col=1
)

# Time interval distribution
fig_storage.add_trace(
    go.Histogram(x=time_intervals, nbinsx=50, name='Time Interval', marker=dict(color='green')),
    row=1, col=2
)

# Storage growth projection (bar chart)
time_periods = ['Hour', 'Day', 'Week', 'Month', 'Year']
storage_amounts = [mb_per_hour, mb_per_day, gb_per_week * 1024, gb_per_month * 1024, gb_per_year * 1024]
storage_units = ['MB', 'MB', 'MB', 'MB', 'MB']

fig_storage.add_trace(
    go.Bar(
        x=time_periods,
        y=storage_amounts,
        name='Storage Required',
        marker=dict(color=['lightblue', 'orange', 'red', 'darkred', 'purple']),
        text=[f'{v:.0f} {u}' if v < 1024 else f'{v/1024:.1f} GB' for v, u in zip(storage_amounts, storage_units)],
        textposition='auto'
    ),
    row=2, col=1
)

# Cumulative file size over time
cumulative_sizes = np.cumsum(line_sizes) / (1024 * 1024)  # Convert to MB
snapshot_times_relative = [(s['ts'] - snapshots[0]['ts']) / 60 for s in snapshots]  # Minutes from start

fig_storage.add_trace(
    go.Scatter(
        x=snapshot_times_relative,
        y=cumulative_sizes,
        mode='lines',
        name='Cumulative Size',
        line=dict(color='purple', width=2)
    ),
    row=2, col=2
)

# Update axes
fig_storage.update_xaxes(title_text='Line Size (bytes)', row=1, col=1)
fig_storage.update_xaxes(title_text='Time Interval (seconds)', row=1, col=2)
fig_storage.update_xaxes(title_text='Time Period', row=2, col=1)
fig_storage.update_xaxes(title_text='Time (minutes)', row=2, col=2)

fig_storage.update_yaxes(title_text='Frequency', row=1, col=1)
fig_storage.update_yaxes(title_text='Frequency', row=1, col=2)
fig_storage.update_yaxes(title_text='Storage (MB)', row=2, col=1, type='log')
fig_storage.update_yaxes(title_text='Cumulative Size (MB)', row=2, col=2)

# Update layout
fig_storage.update_layout(
    height=900,
    showlegend=False,
    template='plotly_white',
    title_text=f'{trading_pair} Storage Performance Metrics'
)

fig_storage.show()

In [ ]:
import os

# Get file information
file_size_bytes = os.path.getsize(file_path)
file_size_kb = file_size_bytes / 1024
file_size_mb = file_size_kb / 1024

# Calculate line statistics
with open(file_path, 'r') as f:
    lines = [line for line in f if line.strip()]
    line_sizes = [len(line.encode('utf-8')) for line in lines]

num_lines = len(lines)
avg_line_size = np.mean(line_sizes)
median_line_size = np.median(line_sizes)
min_line_size = np.min(line_sizes)
max_line_size = np.max(line_sizes)

# Calculate time intervals between snapshots
time_intervals = []
for i in range(1, len(snapshots)):
    interval = snapshots[i]['ts'] - snapshots[i-1]['ts']
    time_intervals.append(interval)

avg_interval = np.mean(time_intervals)
median_interval = np.median(time_intervals)
min_interval = np.min(time_intervals)
max_interval = np.max(time_intervals)

# Calculate sampling rate
duration_seconds = snapshots[-1]['ts'] - snapshots[0]['ts']
duration_minutes = duration_seconds / 60
duration_hours = duration_minutes / 60
sampling_rate_per_second = num_lines / duration_seconds if duration_seconds > 0 else 0
sampling_rate_per_minute = num_lines / duration_minutes if duration_minutes > 0 else 0

# Storage projections
bytes_per_second = file_size_bytes / duration_seconds if duration_seconds > 0 else 0
bytes_per_minute = bytes_per_second * 60
bytes_per_hour = bytes_per_minute * 60
bytes_per_day = bytes_per_hour * 24

mb_per_hour = bytes_per_hour / (1024 * 1024)
mb_per_day = bytes_per_day / (1024 * 1024)
gb_per_day = mb_per_day / 1024
gb_per_week = gb_per_day * 7
gb_per_month = gb_per_day * 30
gb_per_year = gb_per_day * 365

# Create performance dashboard
print("=" * 80)
print("STORAGE PERFORMANCE ANALYSIS")
print("=" * 80)

print("\n📊 FILE STATISTICS")
print(f"  File Path:           {file_path}")
print(f"  Total File Size:     {file_size_bytes:,} bytes ({file_size_kb:.2f} KB / {file_size_mb:.2f} MB)")
print(f"  Number of Lines:     {num_lines:,}")
print(f"  Number of Snapshots: {len(snapshots):,}")

print("\n📏 LINE SIZE STATISTICS")
print(f"  Average Line Size:   {avg_line_size:.2f} bytes")
print(f"  Median Line Size:    {median_line_size:.0f} bytes")
print(f"  Min Line Size:       {min_line_size} bytes")
print(f"  Max Line Size:       {max_line_size} bytes")
print(f"  Std Dev:             {np.std(line_sizes):.2f} bytes")

print("\n⏱️  SAMPLING INTERVAL STATISTICS")
print(f"  Average Interval:    {avg_interval:.3f} seconds")
print(f"  Median Interval:     {median_interval:.3f} seconds")
print(f"  Min Interval:        {min_interval:.3f} seconds")
print(f"  Max Interval:        {max_interval:.3f} seconds")
print(f"  Std Dev:             {np.std(time_intervals):.3f} seconds")

print("\n🔢 SAMPLING RATE")
print(f"  Duration:            {duration_seconds:.2f} seconds ({duration_minutes:.2f} minutes / {duration_hours:.2f} hours)")
print(f"  Snapshots/Second:    {sampling_rate_per_second:.3f}")
print(f"  Snapshots/Minute:    {sampling_rate_per_minute:.2f}")

print("\n💾 STORAGE RATE")
print(f"  Bytes/Second:        {bytes_per_second:,.2f}")
print(f"  KB/Minute:           {bytes_per_minute / 1024:.2f}")
print(f"  MB/Hour:             {mb_per_hour:.2f}")

print("\n📈 STORAGE PROJECTIONS (Continuous Recording)")
print(f"  Per Hour:            {mb_per_hour:.2f} MB")
print(f"  Per Day:             {mb_per_day:.2f} MB ({gb_per_day:.3f} GB)")
print(f"  Per Week:            {gb_per_week:.2f} GB")
print(f"  Per Month (30d):     {gb_per_month:.2f} GB")
print(f"  Per Year:            {gb_per_year:.2f} GB ({gb_per_year/1024:.2f} TB)")

print("\n⚠️  SPACE RISK ASSESSMENT")
if gb_per_day < 1:
    risk = "LOW"
    color = "🟢"
    recommendation = "Current storage rate is sustainable for long-term recording."
elif gb_per_day < 10:
    risk = "MEDIUM"
    color = "🟡"
    recommendation = "Monitor disk space regularly. Consider data rotation or compression."
else:
    risk = "HIGH"
    color = "🔴"
    recommendation = "High storage consumption. Implement data rotation, compression, or sampling reduction."

print(f"  Risk Level:          {color} {risk}")
print(f"  Recommendation:      {recommendation}")

print("\n💡 OPTIMIZATION SUGGESTIONS")
# Calculate compression potential (typical JSON compression ratio is 3-5x)
estimated_compressed_size = file_size_mb / 4  # Assuming 4x compression
print(f"  Estimated with compression (gzip): {estimated_compressed_size:.2f} MB (~{gb_per_day/4:.3f} GB/day)")

# Calculate if reducing depth could help
print(f"  Current depth per snapshot: {len(snapshots[0]['bids'])} levels")
print(f"  Reducing to 10 levels: ~{(gb_per_day * 10) / len(snapshots[0]['bids']):.3f} GB/day")
print(f"  Reducing to 20 levels: ~{(gb_per_day * 20) / len(snapshots[0]['bids']):.3f} GB/day")

print("\n" + "=" * 80)

## Storage Performance Analysis
Analyze file size, storage efficiency, and potential space requirements for long-term data collection.